# Connect database

In [11]:
import duckdb
import time
from pathlib import Path

DB_PATH = "./mimiciv.duckdb"
MIMIC_PATH = Path("../mimic/physionet.org/files/mimiciv/3.1")

HOSP = MIMIC_PATH / "hosp"
ICU = MIMIC_PATH / "icu"

# Only the tables we actually use in this project
TABLES = {
    # Hosp module: demographics, admissions, and labs
    "hosp.patients": HOSP / "patients.csv.gz",
    "hosp.admissions": HOSP / "admissions.csv.gz",
    "hosp.labevents": HOSP / "labevents.csv.gz",
    "hosp.d_labitems": HOSP / "d_labitems.csv.gz",
    # ICU module: stays, charted vitals, and IV medications
    "icu.icustays": ICU / "icustays.csv.gz",
    "icu.chartevents": ICU / "chartevents.csv.gz",
    "icu.d_items": ICU / "d_items.csv.gz",
    "icu.inputevents": ICU / "inputevents.csv.gz",
}


def load_tables(con):
    con.execute("CREATE SCHEMA IF NOT EXISTS hosp")
    con.execute("CREATE SCHEMA IF NOT EXISTS icu")
    for table_name, file_path in TABLES.items():
        if not file_path.exists():
            print(f"  SKIP {table_name}: file not found at {file_path}")
            continue
        print(f"  Loading {table_name}...", end=" ", flush=True)
        start = time.time()
        con.execute(f"DROP TABLE IF EXISTS {table_name}")
        con.execute(f"""
            CREATE TABLE {table_name} AS
            SELECT * FROM read_csv_auto('{file_path}', header=True)
        """)
        n_rows = con.execute(f"SELECT COUNT(*) FROM {table_name}").fetchone()[0]
        elapsed = time.time() - start
        print(f"{n_rows:,} rows in {elapsed:.1f}s")


def create_views(con):
    # Joins patients with admissions; adds hospital LOS in hours
    con.execute("""
        CREATE OR REPLACE VIEW patient_admissions AS
        SELECT
            p.subject_id, p.gender, p.anchor_age, p.anchor_year, p.dod,
            a.hadm_id, a.admittime, a.dischtime, a.deathtime,
            a.admission_type, a.admission_location, a.discharge_location,
            a.insurance, a.language, a.marital_status, a.race,
            a.hospital_expire_flag,
            EXTRACT(EPOCH FROM (a.dischtime - a.admittime)) / 3600 AS hospital_los_hours
        FROM hosp.patients p
        JOIN hosp.admissions a ON p.subject_id = a.subject_id
    """)

    # Enriches ICU stays with demographics from patient_admissions; adds ICU LOS in hours
    con.execute("""
        CREATE OR REPLACE VIEW icu_stays_full AS
        SELECT
            i.stay_id, i.subject_id, i.hadm_id,
            i.first_careunit, i.last_careunit, i.intime, i.outtime,
            EXTRACT(EPOCH FROM (i.outtime - i.intime)) / 3600 AS icu_los_hours,
            pa.gender, pa.anchor_age, pa.race, pa.insurance,
            pa.admission_type, pa.hospital_expire_flag, pa.dod
        FROM icu.icustays i
        JOIN patient_admissions pa ON i.hadm_id = pa.hadm_id
    """)

    # Filters chartevents to CAM/delirium assessments only; normalises values to binary 0/1
    con.execute("""
        CREATE OR REPLACE VIEW cam_icu_assessments AS
        SELECT
            ce.subject_id, ce.hadm_id, ce.stay_id, ce.charttime, ce.itemid,
            di.label AS item_label, ce.value, ce.valuenum,
            CASE 
                WHEN ce.value IN ('Positive', 'Yes', 'positive', 'yes') THEN 1
                WHEN ce.value IN ('Negative', 'No', 'negative', 'no') THEN 0
                ELSE NULL
            END AS cam_positive
        FROM icu.chartevents ce
        JOIN icu.d_items di ON ce.itemid = di.itemid
        WHERE LOWER(di.label) LIKE '%cam%'
           OR LOWER(di.label) LIKE '%delirium%'
           OR LOWER(di.label) LIKE '%confusion assessment%'
    """)

    # Aggregates CAM assessments per ICU stay: total/positive counts, ever-delirious flag, first/last assessment times
    con.execute("""
        CREATE OR REPLACE VIEW delirium_summary AS
        SELECT
            stay_id,
            COUNT(*) AS total_assessments,
            SUM(cam_positive) AS positive_assessments,
            MAX(cam_positive) AS ever_delirious,
            MIN(CASE WHEN cam_positive = 1 THEN charttime END) AS first_positive_time,
            MIN(charttime) AS first_assessment_time,
            MAX(charttime) AS last_assessment_time
        FROM cam_icu_assessments
        WHERE cam_positive IS NOT NULL
        GROUP BY stay_id
    """)

    # Joins lab results with item labels and categories for human-readable lab names and units
    con.execute("""
        CREATE OR REPLACE VIEW labs_labeled AS
        SELECT
            le.subject_id, le.hadm_id, le.charttime, le.itemid,
            dl.label AS lab_name, dl.category AS lab_category,
            le.value, le.valuenum, le.valueuom AS unit, le.flag
        FROM hosp.labevents le
        JOIN hosp.d_labitems dl ON le.itemid = dl.itemid
    """)


# Build only if the database doesn't exist; otherwise just connect
if Path(DB_PATH).exists():
    print(f"Database already exists at {DB_PATH}, connecting...")
    con = duckdb.connect(database=DB_PATH)
else:
    print(f"No database found at {DB_PATH}, building from raw files...")
    if not MIMIC_PATH.exists():
        raise FileNotFoundError(
            f"Raw MIMIC-IV files not found at {MIMIC_PATH}. "
            f"Update MIMIC_PATH at the top of this cell to point to your data."
        )
    con = duckdb.connect(database=DB_PATH)
    print("\n>>> Loading tables...")
    load_tables(con)
    print("\n>>> Creating views...")
    create_views(con)
    print(f"\nDatabase built at {DB_PATH}")

print("Connection ready.")

Database already exists at ./mimiciv.duckdb, connecting...
Connection ready.


# Build cohort

In [13]:
MIN_AGE = 18
MIN_LOS_HOURS = 24

# Build the cohort: adult ICU stays with at least one labelable CAM-ICU assessment
con.execute(f"""
    CREATE OR REPLACE TABLE cohort AS
    SELECT 
        s.stay_id,
        s.subject_id,
        s.hadm_id,
        s.first_careunit,
        s.intime,
        s.outtime,
        s.icu_los_hours,
        s.gender,
        s.anchor_age AS age,
        s.race,
        s.admission_type,
        s.hospital_expire_flag
    FROM icu_stays_full s
    WHERE s.anchor_age >= {MIN_AGE}
      AND s.icu_los_hours >= {MIN_LOS_HOURS}
      AND s.stay_id IN (
          SELECT stay_id FROM delirium_summary 
          WHERE ever_delirious IS NOT NULL
      )
""")

# Check the size of the cohort
n_stays = con.execute("SELECT COUNT(*) FROM cohort").fetchone()[0]
n_patients = con.execute("SELECT COUNT(DISTINCT subject_id) FROM cohort").fetchone()[0]
print(f"Cohort: {n_stays:,} ICU stays, {n_patients:,} unique patients")

Cohort: 59,432 ICU stays, 43,923 unique patients


## Label distribution and demographics

In [15]:
# How many cohort stays were ever delirious vs not?
labels = con.execute("""
    SELECT 
        COUNT(*) AS n_stays,
        SUM(CASE WHEN ds.ever_delirious = 1 THEN 1 ELSE 0 END) AS n_delirious,
        SUM(CASE WHEN ds.ever_delirious = 0 THEN 1 ELSE 0 END) AS n_not_delirious
    FROM cohort c
    JOIN delirium_summary ds ON c.stay_id = ds.stay_id
""").df()
print("Label distribution:")
print(labels.to_string(index=False))

# Quick demographic summary
demo = con.execute("""
    SELECT gender, COUNT(*) AS n,
    ROUND(AVG(age), 1) AS mean_age
    FROM cohort
    GROUP BY gender
""").df()
print("\nBy gender:")
print(demo.to_string(index=False))

race = con.execute("""
    SELECT race, COUNT(*) AS n
    FROM cohort
    GROUP BY race
    ORDER BY n DESC
    LIMIT 10
""").df()
print("\nTop races:")
print(race.to_string(index=False))

Label distribution:
 n_stays  n_delirious  n_not_delirious
   59432      30449.0          28983.0

By gender:
gender     n  mean_age
     F 25779      64.0
     M 33653      62.7

Top races:
                          race     n
                         WHITE 35917
                       UNKNOWN  6116
        BLACK/AFRICAN AMERICAN  5288
                         OTHER  2221
        WHITE - OTHER EUROPEAN  1709
              UNABLE TO OBTAIN  1028
HISPANIC/LATINO - PUERTO RICAN   858
               ASIAN - CHINESE   702
               WHITE - RUSSIAN   676
                         ASIAN   648


## Save cohort to file

In [17]:
import os

os.makedirs("./data", exist_ok=True)

# Save the cohort so we don't have to recompute
cohort_df = con.execute("SELECT * FROM cohort").df()
cohort_df.to_parquet("./data/cohort.parquet", index=False, engine="fastparquet")
print(f"Saved cohort: {cohort_df.shape}")

Saved cohort: (59432, 12)


# Feature extraction

## Define itemids

In [18]:
# Vital sign itemids (from icu.chartevents)
VITAL_ITEMIDS = {
    "heart_rate": [220045],
    "sbp": [220050, 220179],  # arterial + non-invasive systolic
    "dbp": [220051, 220180],
    "mbp": [220052, 220181],
    "resp_rate": [220210],
    "spo2": [220277],
    "temp_c": [223762],
    "temp_f": [223761],  # Fahrenheit, will convert later
    "gcs_eye": [220739],
    "gcs_verbal": [223900],
    "gcs_motor": [223901],
}

# Lab itemids (from hosp.labevents)
LAB_ITEMIDS = {
    "sodium": [50983],
    "potassium": [50971],
    "chloride": [50902],
    "bicarbonate": [50882],
    "bun": [51006],
    "creatinine": [50912],
    "glucose": [50931],
    "hemoglobin": [51222],
    "hematocrit": [51221],
    "wbc": [51301],
    "platelets": [51265],
    "lactate": [50813],
}

# Medication itemids (from icu.inputevents)
MED_ITEMIDS = {
    "benzodiazepine": [221668, 221385, 221623],
    "opioid": [221744, 225942, 221833, 225154],
    "propofol": [222168],
    "dexmedetomidine": [229420, 225150],
    "antipsychotic": [221824],
}

print("Vitals:", list(VITAL_ITEMIDS.keys()))
print("Labs:", list(LAB_ITEMIDS.keys()))
print("Meds:", list(MED_ITEMIDS.keys()))

Vitals: ['heart_rate', 'sbp', 'dbp', 'mbp', 'resp_rate', 'spo2', 'temp_c', 'temp_f', 'gcs_eye', 'gcs_verbal', 'gcs_motor']
Labs: ['sodium', 'potassium', 'chloride', 'bicarbonate', 'bun', 'creatinine', 'glucose', 'hemoglobin', 'hematocrit', 'wbc', 'platelets', 'lactate']
Meds: ['benzodiazepine', 'opioid', 'propofol', 'dexmedetomidine', 'antipsychotic']


## Extract vitals and aggregate into 6-hour windows

In [ ]:
WINDOW_HOURS = 6
MAX_WINDOWS = 28 # 28 windows of 6 hours = 7 days, which covers 95% of delirium onset times in our cohort

# Build a row for each (stay, window) pair
con.execute(f"""
    CREATE OR REPLACE TABLE windows AS
    SELECT 
        c.stay_id,
        c.subject_id,
        c.hadm_id,
        c.intime,
        w.window_idx,
        c.intime + INTERVAL (w.window_idx * {WINDOW_HOURS}) HOUR AS window_start,
        c.intime + INTERVAL ((w.window_idx + 1) * {WINDOW_HOURS}) HOUR AS window_end
    FROM cohort c
    CROSS JOIN (
        SELECT generate_series AS window_idx 
        FROM generate_series(0, {MAX_WINDOWS - 1})
    ) w
    WHERE c.intime + INTERVAL (w.window_idx * {WINDOW_HOURS}) HOUR < c.outtime
""")

n_windows = con.execute("SELECT COUNT(*) FROM windows").fetchone()[0]
print(f"Built {n_windows:,} windows")

Built 819,563 windows
